# Ingesta de datos

In [1]:
import pandas as pd
import re
import unicodedata
# Cargue de datos

# costos por servicio costeados
df_costeos_serv_raw1 = pd.read_json('response.json')
df_costeos_serv_raw1.columns = ['id_costeoxpto', 'id_serv', 'costo_antes_iva', 'fecha_creacion']

df_costeos_serv_raw2 = pd.read_json('response2.json')
df_costeos_serv_raw2.columns = ['id_costeoxpto', 'id_serv', 'costo_antes_iva', 'fecha_creacion']

df_costeos_serv_raw3 = pd.read_json('response3.json')
df_costeos_serv_raw3.columns = ['id_costeoxpto', 'id_serv', 'costo_antes_iva', 'fecha_creacion']

df_costeos_serv_raw = pd.concat([df_costeos_serv_raw1, df_costeos_serv_raw2, df_costeos_serv_raw3], ignore_index=True)

# mapa de servicios id
df_servicios_id = pd.read_csv('Todos Procesos Producción.csv')

# Costeos x PTO
df_costeos_pto = pd.read_csv('Todos Costeo Producto.csv')

# OP detalle
df_op_det = pd.read_csv('File.csv')

# Ordenes de Servicio
df_os = pd.read_csv('Todas OS.csv')

#Comercial
df_comercial = pd.read_csv('Todos Costeos.csv')

# Data Cleaning

In [2]:
# Homologación servicios costeos
df_costeos_serv = df_costeos_serv_raw.merge(
    df_servicios_id[["Id.", "Proceso Producción"]].rename(columns={"Id.": "id_serv"}),
    on="id_serv",
    how="left",
).drop(columns=["id_serv"])

In [3]:
df_costeos_serv.drop_duplicates(inplace=True)

In [4]:
# union con costeos x pto
df_costeos_pto = df_costeos_pto[
    ["Costeo_Producto", "Producto", "Cliente", "Cantidad Solicitada", "Id.", "Fecha"]
].rename(columns={"Producto":"Productos"})

df_costeos_pto = df_costeos_pto.merge(
    df_costeos_serv.rename(columns={"id_costeoxpto": "Id."}),
    on="Id.",
    how="left",
)
df_costeos_pto


,Costeo_Producto,Productos,Cliente,Cantidad Solicitada,Id.,Fecha,costo_antes_iva,fecha_creacion,Proceso Producción
0,COS-0012960-1,CHAQUETA SPORT,MOLT,60,83563364,17/04/2026,16500.0,2026-04-17T20:35:36Z,Confección
1,COS-0012960-1,CHAQUETA SPORT,MOLT,60,83563364,17/04/2026,1800.0,2026-04-17T20:34:48Z,Bordado
2,COS-0012960-2,CHAQUETA SPORT,MOLT,60,83563935,17/04/2026,18000.0,2026-04-17T20:38:59Z,Confección
3,COS-0012960-2,CHAQUETA SPORT,MOLT,60,83563935,17/04/2026,1800.0,2026-04-17T20:38:59Z,Bordado
4,COS-0012960-3,CHAQUETA SPORT,MOLT,60,83564293,17/04/2026,16500.0,2026-04-17T20:49:30Z,Confección
...,...,...,...,...,...,...,...,...,...
19801,COS-0011567-13,Camibuso Polo Manga Larga,AECSA,1,73229251,06/10/2025,1200.0,2025-10-06T13:36:29Z,Bordado
19802,COS-0011567-14,Camibuso Polo Manga Larga,AECSA,1,73229258,06/10/2025,5000.0,2025-10-06T13:36:33Z,Confección
19803,COS-0011567-14,Camibuso Polo Manga Larga,AECSA,1,73229258,06/10/2025,1200.0,2025-10-06T13:36:33Z,Bordado
19804,COS-0011567-15,Camibuso Polo Manga Larga,AECSA,1,73229265,06/10/2025,5000.0,2025-10-06T13:36:37Z,Confección


In [5]:
df_costeos_pto.duplicated().sum()

np.int64(0)

In [6]:
# unir a OP detalle
df_op_det.drop(
    columns=[
        "ClienteNombre",
        "Categoría Proc",
        "Detalle",
        "Colores",
        "Total Pieza",
        "Cantidad Remitida",
        "Precio Unitario Sin IVA",
        "Estado OP",
        "Fecha Promesa",
        "Ficha Técnica Producto",
        "Muestra",
        "Id.",
        "OS",
    ],
    inplace=True,
)

df_op_det = df_op_det.rename(columns={'Id Costeo Producto' : 'Id.'})


df_op_det = df_op_det.merge(
    df_costeos_pto,
    on=["Costeo_Producto", "Productos", "Id."],
    how="left",
)

# filtro de ultimos 6 meses
df_op_det = df_op_det[df_op_det["Fecha"] >= "2025-10-01"]

df_op_det.duplicated().sum()


np.int64(6)

In [7]:
rev_fechas = df_op_det.groupby('OP Det')['Fecha'].nunique()

dupf=rev_fechas[rev_fechas > 1]

print(dupf)

Series([], Name: Fecha, dtype: int64)


In [8]:
res = df_comercial.groupby('Costeo')['Nombre del Comercial'].nunique()

dup = res[res > 1]

print(dup)

Costeo
COS-0011966    2
COS-0011994    2
COS-0012200    2
COS-0012208    2
COS-0012403    2
Name: Nombre del Comercial, dtype: int64


In [9]:
# Limpieza de comercial

df_comercial.columns

# Eliminación de columnas no relevantes
df_comercial.drop(
    columns=[
        "Nombre",
        "Estado",
        "Margen Ponderado",
        "Creado en",
        "Acciones",
    ],
    inplace=True,
)
df_comercial.head()


,Costeo,Cliente,Nombre del Comercial,Id.
0,COS-0012960,MOLT,Xiomara Aranda Perea,83563363
1,COS-0012959,Molt Panamá (PA),Xiomara Aranda Perea,83557637
2,COS-0012958,Joe Hawkins,Simón Román,83541478
3,COS-0012957,Sodimac,Pilar Gómez,83506298
4,COS-0012957,G4S,Pilar Gómez,83506013


In [10]:
df_op_det['OP-D'] = df_op_det['Num-OP'].astype(str) + '-' + df_op_det['Secuencia'].astype(int).astype(str)

In [11]:
# añadir prefijo OS a columnas de df_os
for col in df_os.columns:
    if col not in ['Num OS', 'Orden de Satélite', 'Satélite Procesos']:
        df_os.rename(columns={col: 'OS-' + col}, inplace=True)

df_os['OP-D'] = df_os['Orden de Satélite'].str.split('-').str[2] + '-' + df_os['Orden de Satélite'].str.split('-').str[3]

df_os_grouped = df_os.groupby(['OP-D', 'Satélite Procesos']).apply(
    lambda x: pd.Series({
        'OS-Valor Unitario': (x['OS-Valor Unitario'] * x['OS-Cantidad']).sum() / x['OS-Cantidad'].sum(),
        'OS-Cantidad': x['OS-Cantidad'].sum(),
        'Num OS': tuple(x['Num OS'].unique())
    })
).reset_index()

C:\Users\Laura\AppData\Local\Temp\ipykernel_26048\2228398441.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_os_grouped = df_os.groupby(['OP-D', 'Satélite Procesos']).apply(


In [12]:
df_os_grouped.duplicated().sum()    

np.int64(0)

In [13]:
# Costeo servicio op vs os
df_op_det_vs_os = df_op_det.merge(
    df_os_grouped.rename(columns={"Satélite Procesos": "Proceso Producción"}),
    on=["OP-D", "Proceso Producción"],
    how="left",
)

In [14]:
# Filtramos para quedarnos solo con las filas que no esten vacias en OS-Valor Unitario 
df_op_det_vs_os = df_op_det_vs_os[df_op_det_vs_os['OS-Valor Unitario'].notna()]

# Separamos el último número del costeo para poder unir con el df de comercial
df_op_det_vs_os['Costeo'] = (df_op_det_vs_os['Costeo_Producto'].str.rsplit('-', n=1).str[0])


In [15]:
df_comercial.columns

Index(['Costeo', 'Cliente', 'Nombre del Comercial', 'Id.'], dtype='object')

In [16]:
df_final = df_op_det_vs_os.merge(
    df_comercial,
    on=["Costeo", "Cliente"],
    how="left"
)
df_final


,Num-OP,Secuencia,OP Det,Productos,Producir,Total Precio Sin IVA,Creado El,Id._x,Costeo_Producto,Cliente,...,costo_antes_iva,fecha_creacion,Proceso Producción,OP-D,OS-Valor Unitario,OS-Cantidad,Num OS,Costeo,Nombre del Comercial,Id._y
0,6698,1.0,6698-1-POLO MANGA LARGA,POLO MANGA LARGA,19,491606.0,30/03/2026 11:17,80671827,COS-0012655-2,AECSA,...,1200.0,2026-02-25T17:36:29Z,Bordado,6698-1,1200.0,19.0,"(21625, 21624)",COS-0012655,Xiomara Aranda Perea,80671826.0
1,6698,2.0,6698-2-POLO MANGA LARGA,POLO MANGA LARGA,5,127765.0,30/03/2026 11:17,80671876,COS-0012655-2,AECSA,...,1200.0,2026-02-25T17:40:05Z,Bordado,6698-2,1200.0,5.0,"(21626,)",COS-0012655,Xiomara Aranda Perea,80671826.0
2,6696,10.0,6696-10-Gorra seis cascos,Gorra seis cascos,90,1249830.0,29/03/2026 9:48,82496900,COS-0012874-13,NUTRESA,...,1800.0,2026-03-28T16:35:29Z,Bordado,6696-10,2500.0,90.0,"(21589,)",COS-0012874,Luis Guzman,82496746.0
3,6696,11.0,6696-11-Gorra seis cascos,Gorra seis cascos,20,277740.0,29/03/2026 9:48,82496904,COS-0012874-14,NUTRESA,...,1800.0,2026-03-28T16:35:29Z,Bordado,6696-11,2500.0,20.0,"(21590,)",COS-0012874,Luis Guzman,82496746.0
4,6696,12.0,6696-12-Gorra seis cascos,Gorra seis cascos,135,1874745.0,29/03/2026 9:48,82496916,COS-0012874-16,NUTRESA,...,1800.0,2026-03-28T16:35:29Z,Bordado,6696-12,2500.0,135.0,"(21591,)",COS-0012874,Luis Guzman,82496746.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
539,6363,1.0,6363-1-camisa,camisa,2,129408.0,21/10/2025 16:53,74081481,COS-0011966-12,Hoteles Decameron,...,12000.0,2025-10-21T21:30:26Z,Confección,6363-1,11500.0,2.0,"(19953,)",COS-0011966,Angela Jaramillo,74081365.0
540,6363,2.0,6363-2-Pantalón dotación,Pantalón dotación,2,133356.0,21/10/2025 16:53,74081540,COS-0011966-15,Hoteles Decameron,...,1200.0,2025-10-21T21:30:26Z,Bordado,6363-2,1100.0,2.0,"(20117,)",COS-0011966,Angela Jaramillo,74081703.0
541,6363,2.0,6363-2-Pantalón dotación,Pantalón dotación,2,133356.0,21/10/2025 16:53,74081540,COS-0011966-15,Hoteles Decameron,...,1200.0,2025-10-21T21:30:26Z,Bordado,6363-2,1100.0,2.0,"(20117,)",COS-0011966,Angela Jaramillo,74081365.0
542,6362,3.0,6362-3-CAMISA MANGA LARGA,CAMISA MANGA LARGA,6,272886.0,21/10/2025 15:23,74072922,COS-0011917-3,DI-MATIC,...,4000.0,2025-10-21T19:41:36Z,Bordado,6362-3,4000.0,6.0,"(19670,)",COS-0011917,Xiomara Aranda Perea,73662140.0


In [17]:
categorias_oficiales = [
    'Accesorios', 'Bata', 'Bermuda', 'Blusa', 'Bolígrafo', 'Botas', 'Botella', 'Botón', 'Broche', 'Buzo', 'Camibuzo', 
    'Camiseta', 'Camisa', 'Cachuchera', 'Casco', 'Chaleco', 'Chaqueta', 'Cinta', 'Cofia', 'Conjunto', 'Cordón', 
    'Cremallera', 'Cuello', 'Delantal', 'Diseño', 'Embone', 'Falda', 'Gafas seguridad', 'Gorra', 'Gorro', 'Guantes', 
    'Guata', 'Hilo', 'Hoodie', 'Impermeable', 'Jean', 'Libreta', 'Lonchera', 'Mangas', 'Marquilla', 'Máscara', 'Medias', 
    'Morral', 'Ojalete', 'Overol', 'Pantalón', 'Pantaloneta', 'Pañoleta', 'Paraguas', 'Pavas', 'Polainas', 'Resorte', 
    'Ropa interior', 'Saco', 'Sastre', 'Servicios', 'Sesgo', 'Sudadera', 'Suéter', 'Tankas', 'Tapabocas', 
    'Tapaoídos', 'Tecnología', 'Termo', 'Uniformes', 'Velcro', 'Zapatos', 'Escafandra'
]

# Creamos una función para normalizar el texto, eliminando acentos y convirtiendo a minúsculas
def normalizar(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    return texto

# Mapeos especiales: palabras clave -> categoría oficial
MAPEOS_ESPECIALES = {
    'polo':       'Camiseta',
    'camibuso':   'Camibuzo',   # cubre "camibuso" (mal escrito) -> "Camibuzo"
    'harrington': 'Chaqueta',
    'termica':    'Chaqueta',   # cubre "térmica" también (por la normalización)
    'hoddie':     'Hoodie',     # cubre el typo "hoddie" -> "Hoodie"
    'jogger':     'Pantalón',
}

def clasificar(Producto):
    producto_norm = normalizar(Producto)
    
    # 1. Primero revisamos los mapeos especiales
    for keyword, categoria in MAPEOS_ESPECIALES.items():
        pattern = r'\b' + re.escape(keyword) + r'\b'
        if re.search(pattern, producto_norm):
            return categoria
    
    # 2. Luego la búsqueda genérica en categorías oficiales
    for cat in categorias_oficiales:
        cat_norm = normalizar(cat)
        pattern = r'\b' + re.escape(cat_norm) + r'\b'
        if re.search(pattern, producto_norm):
            return cat
    
    return 'Otros'

df_final['Categoría'] = df_final['Productos'].apply(clasificar)

In [18]:
df_final.drop_duplicates
df_final = df_final[[
    'Num-OP', 'Secuencia', 'OP-D', 'OP Det', 'Productos', 'Cliente', 
    'Fecha', 'Id._x', 'Costeo_Producto', 'Costeo', 'Proceso Producción', 
    'Producir', 'OS-Cantidad', 'Num OS',  'Nombre del Comercial', 'Categoría',
    'Total Precio Sin IVA', 'costo_antes_iva', 'OS-Valor Unitario', 
]].rename(columns={'Id._x' : 'Id.'})

df_final.drop_duplicates(inplace=True)

In [19]:
df_final[df_final.duplicated(keep=False)]

,Num-OP,Secuencia,OP-D,OP Det,Productos,Cliente,Fecha,Id.,Costeo_Producto,Costeo,Proceso Producción,Producir,OS-Cantidad,Num OS,Nombre del Comercial,Categoría,Total Precio Sin IVA,costo_antes_iva,OS-Valor Unitario


In [20]:
df_final.to_excel('df_finall.xlsx')